# Notebook 1 — Symbolic DQN: Train on World 1-1

Trains a **DQN** agent on World 1-1 using the RAM-based symbolic grid as observations. This is the first baseline: the simplest RL algorithm applied to the most compact observation space.

## Study Progression
1. **→ Symbolic DQN — World 1-1 (you are here)**
2. Pixel PPO — World 1-1
3. Symbolic PPO — World 1-1
4. Transfer Learning — Symbolic PPO 1-1 → 1-2
5. World 1 Full — Random Starts + All 4 Levels

## Setup
- **Observation:** flattened 13×16 symbolic grid × 4 frame-stack + power state = **833-dim** vector
- **Policy:** `MlpPolicy` with default net arch
- **Training:** 8 parallel envs, 2M steps
- **Output:** `models/symbolic_dqn_w1l1/`

In [ ]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
    !git pull
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

In [ ]:
import torch
from stable_baselines3 import DQN

from src.wrappers import make_symbolic_vec_env, make_symbolic_env
from src.utils.callbacks import CheckpointAndLogCallback
from src.config import DQNConfig

print(f'Using CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Create 8 parallel symbolic environments for World 1-1
NUM_ENVS = 8

env = make_symbolic_vec_env(
    env_id='SuperMarioBros-1-1-v3',
    skip=4,
    n_stack=4,
    flatten=True,
    num_envs=NUM_ENVS,
)
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: {env.action_space.n}')
print(f'Parallel envs: {NUM_ENVS}')

In [ ]:
# Configure DQN with symbolic (MlpPolicy) settings
LOG_DIR = 'logs/symbolic_dqn_w1l1'
SAVE_DIR = 'models/symbolic_dqn_w1l1'

config = DQNConfig(
    policy='MlpPolicy',
    total_timesteps=2_000_000,
)

model = DQN(
    policy=config.policy,
    env=env,
    learning_rate=config.learning_rate,
    buffer_size=config.buffer_size,
    learning_starts=config.learning_starts,
    batch_size=config.batch_size,
    gamma=config.gamma,
    target_update_interval=config.target_update_interval,
    exploration_fraction=config.exploration_fraction,
    exploration_initial_eps=config.exploration_initial_eps,
    exploration_final_eps=config.exploration_final_eps,
    train_freq=config.train_freq,
    tensorboard_log=LOG_DIR,
    verbose=1,
    device='cpu',  # MLP is faster on CPU
)

print(f'Total timesteps: {config.total_timesteps:,}')
print(f'Device: {model.device}')
print(f'Buffer size: {config.buffer_size:,}')
print(f'Learning starts: {config.learning_starts:,}')

In [ ]:
# Launch TensorBoard (Colab inline)
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

In [ ]:
# Train for 2M steps
callback = CheckpointAndLogCallback(
    save_path=SAVE_DIR,
    save_freq=50_000,
)

model.learn(
    total_timesteps=config.total_timesteps,
    callback=callback,
    log_interval=10,
)
model.save(f'{SAVE_DIR}/final_model')
print('Training complete — final_model saved!')

In [ ]:
# Plot training curves from callback
import matplotlib.pyplot as plt
import numpy as np

all_rewards = callback.episode_rewards
all_lengths = callback.episode_lengths
all_flags   = callback.episode_flags

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
window = min(100, len(all_rewards))

for ax, data, color, title in zip(
    axes,
    [all_rewards, all_lengths, None],
    ['blue', 'orange', 'green'],
    ['Episode Rewards', 'Episode Lengths', 'Cumulative Flag Rate'],
):
    if title == 'Cumulative Flag Rate':
        if all_flags:
            cum = np.cumsum(all_flags) / (np.arange(len(all_flags)) + 1)
            ax.plot(cum, color=color, linewidth=2)
            ax.set_ylim(0, 1)
    else:
        ax.plot(data, alpha=0.3, color=color)
        if len(data) > window:
            smoothed = np.convolve(data, np.ones(window) / window, mode='valid')
            ax.plot(range(window - 1, len(data)), smoothed, color=color, linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('Episode')

plt.tight_layout()
os.makedirs(SAVE_DIR, exist_ok=True)
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=150)
plt.show()

In [ ]:
# Retrieve TensorBoard metrics programmatically
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import glob

event_files = sorted(glob.glob(f'{LOG_DIR}/**/events.out.tfevents.*', recursive=True))
print(f'Found {len(event_files)} TensorBoard event file(s)')

if event_files:
    ea = EventAccumulator(os.path.dirname(event_files[-1]))
    ea.Reload()

    print('\nAvailable scalar tags:')
    for tag in ea.Tags()['scalars']:
        print(f'  - {tag}')

    metrics = {}
    for tag in ea.Tags()['scalars']:
        events = ea.Scalars(tag)
        metrics[tag] = {
            'steps': [e.step for e in events],
            'values': [e.value for e in events],
        }

    tb_keys = [
        'rollout/ep_rew_mean',
        'rollout/ep_len_mean',
        'rollout/flag_rate_100',
        'rollout/exploration_rate',
        'train/loss',
        'train/n_updates',
    ]
    available_keys = [k for k in tb_keys if k in metrics]

    if available_keys:
        n_plots = len(available_keys)
        cols = min(3, n_plots)
        rows = (n_plots + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
        if n_plots == 1:
            axes = [axes]
        else:
            axes = np.array(axes).flatten()

        for ax, key in zip(axes, available_keys):
            ax.plot(metrics[key]['steps'], metrics[key]['values'], linewidth=1.5)
            ax.set_title(key)
            ax.set_xlabel('Timesteps')
            ax.grid(True, alpha=0.3)

        for ax in axes[len(available_keys):]:
            ax.set_visible(False)

        plt.tight_layout()
        plt.savefig(f'{SAVE_DIR}/tensorboard_metrics.png', dpi=150)
        plt.show()
    else:
        print('No matching scalar tags found.')

In [ ]:
# Evaluate final model — 10 episodes
import numpy as np
from stable_baselines3 import DQN

eval_model = DQN.load(f'{SAVE_DIR}/final_model')

eval_env = make_symbolic_env(
    env_id='SuperMarioBros-1-1-v3',
    skip=4, n_stack=4, flatten=True,
)

rewards, lengths, flags = [], [], []

for ep in range(10):
    reset_result = eval_env.reset()
    obs = reset_result[0] if isinstance(reset_result, tuple) else reset_result
    done, total_reward, steps, flag = False, 0.0, 0, False

    while not done:
        action, _ = eval_model.predict(obs, deterministic=True)
        result = eval_env.step(int(action))
        if len(result) == 5:
            obs, reward, terminated, truncated, info = result
            done = terminated or truncated
        else:
            obs, reward, done, info = result
        total_reward += float(reward)
        steps += 1
        if isinstance(info, dict) and info.get('flag_get', False):
            flag = True

    rewards.append(total_reward)
    lengths.append(steps)
    flags.append(flag)
    print(f'Episode {ep+1:2d}: reward={total_reward:.1f}, steps={steps}, {"FLAG!" if flag else "DEAD"}')

print(f'\nMean reward: {np.mean(rewards):.1f} \u00b1 {np.std(rewards):.1f}')
print(f'Mean length: {np.mean(lengths):.0f}')
print(f'Flag rate:   {np.mean(flags):.0%}')

eval_env.close()

In [ ]:
# Download model from Colab (optional)
if os.getenv('COLAB_RELEASE_TAG'):
    from google.colab import files
    import shutil
    shutil.make_archive('symbolic_dqn_w1l1', 'zip', SAVE_DIR)
    files.download('symbolic_dqn_w1l1.zip')
    print('Model archive downloaded!')